# GNN for Max-Cut: a small label-free example

This notebook shows how a GNN can optimize a discrete graph objective without supervised optimal-solution labels.

For a graph \(G=(V,E)\), Max-Cut partitions nodes into two sets and maximizes the number (or weight) of edges crossing the partition.

The GNN outputs a probability \(p_i\in(0,1)\) for each node. For an edge \((i,j)\), a differentiable approximation of the probability that the edge crosses the cut is

\[
p_i(1-p_j)+(1-p_i)p_j.
\]

We maximize the sum of these soft cut contributions, then round \(p_i\ge 0.5\) to obtain a discrete solution. Because the graph is intentionally small, we can compare against the brute-force optimum.


In [ ]:
import itertools
import random
import numpy as np
import networkx as nx
import torch
from torch import nn
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## 1. Build a small graph

Node features are intentionally simple. The purpose is to demonstrate the optimization pipeline rather than engineer the strongest Max-Cut model.


In [ ]:
n = 18
g = nx.erdos_renyi_graph(n=n, p=0.22, seed=SEED)

# Ensure the graph is connected enough for a meaningful toy example.
if g.number_of_edges() == 0:
    raise RuntimeError("Generated graph has no edges.")

edges = np.array(list(g.edges()), dtype=np.int64)
directed_edges = np.vstack([edges, edges[:, ::-1]])
edge_index = torch.tensor(directed_edges.T, dtype=torch.long)

degree = np.array([g.degree(i) for i in range(n)], dtype=np.float32)
degree = degree / max(degree.max(), 1.0)
x = torch.tensor(np.c_[degree, np.ones(n, dtype=np.float32)], dtype=torch.float32)

data = Data(x=x, edge_index=edge_index).to(device)
print(data)
print("Undirected edges:", g.number_of_edges())


## 2. GCN model and differentiable Max-Cut objective

The model produces one logit per node. We convert logits to probabilities with a sigmoid.


In [ ]:
class MaxCutGCN(nn.Module):
    def __init__(self, in_dim=2, hidden=48):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.head = nn.Linear(hidden, 1)

    def forward(self, graph):
        h = torch.relu(self.conv1(graph.x, graph.edge_index))
        h = torch.relu(self.conv2(h, graph.edge_index))
        return self.head(h).squeeze(-1)


def soft_cut_value(prob, undirected_edges):
    u = torch.tensor(undirected_edges[:, 0], dtype=torch.long, device=prob.device)
    v = torch.tensor(undirected_edges[:, 1], dtype=torch.long, device=prob.device)
    return (prob[u] * (1 - prob[v]) + (1 - prob[u]) * prob[v]).sum()


model = MaxCutGCN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)

for epoch in range(500):
    logits = model(data)
    prob = torch.sigmoid(logits)
    objective = soft_cut_value(prob, edges)
    loss = -objective

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f"epoch={epoch+1:3d} soft_cut={objective.item():.3f}")


## 3. Round the probabilities and evaluate the discrete cut


In [ ]:
@torch.no_grad()
def discrete_solution(model, graph):
    p = torch.sigmoid(model(graph)).cpu().numpy()
    assignment = (p >= 0.5).astype(int)
    return p, assignment


def cut_value(assignment, undirected_edges):
    return int(sum(assignment[u] != assignment[v] for u, v in undirected_edges))


prob, assignment = discrete_solution(model, data)
gnn_value = cut_value(assignment, edges)

print("GNN cut value:", gnn_value)
print("Node probabilities:", np.round(prob, 3))
print("Partition:", assignment)


## 4. Brute-force optimum for validation

Brute force is only feasible because this is a tiny graph. It gives us a trustworthy reference for the educational example.


In [ ]:
def brute_force_maxcut(num_nodes, undirected_edges):
    best_value = -1
    best_assignment = None

    # Fix node 0 to one side because complementary partitions have equal value.
    for tail in itertools.product([0, 1], repeat=num_nodes - 1):
        a = np.array((0,) + tail, dtype=int)
        value = cut_value(a, undirected_edges)
        if value > best_value:
            best_value = value
            best_assignment = a.copy()

    return best_value, best_assignment


opt_value, opt_assignment = brute_force_maxcut(n, edges)
gap_pct = 100.0 * (opt_value - gnn_value) / max(opt_value, 1)

print("Optimal cut value:", opt_value)
print("GNN cut value:", gnn_value)
print(f"Gap: {gap_pct:.2f}%")


## 5. What this example does—and does not—show

This notebook demonstrates:

```text
graph
 -> GCN
 -> node probabilities
 -> differentiable combinatorial objective
 -> rounding
 -> discrete solution
 -> exact small-instance validation
```

It is **not** intended to compete with state-of-the-art Max-Cut solvers.

Natural extensions include GraphSAGE/GIN/GAT, heterophily-aware GNNs, randomized rounding, local search, QUBO formulations, and solver warm starts.
